In [ ]:
# Terminal + Ollama configuration with one-switch profiles for T4
import os

# Choose one profile: "agente", "codigo", "geral", "imagem"
PROFILE = "geral"

PROFILE_PRESETS = {
    "agente": {
        "default_model": "nous-hermes2:10.7b",
        "preload_models": ["nous-hermes2:10.7b"],
        "max_loaded_models": 1,
        "num_parallel_requests": 1
    },
    "codigo": {
        "default_model": "qwen2.5-coder:7b",
        "preload_models": ["qwen2.5-coder:7b", "deepseek-coder:6.7b"],
        "max_loaded_models": 2,
        "num_parallel_requests": 2
    },
    "geral": {
        "default_model": "llama3.1:8b",
        "preload_models": ["llama3.1:8b", "mistral:7b"],
        "max_loaded_models": 2,
        "num_parallel_requests": 2
    },
    "imagem": {
        "default_model": "llama3.1:8b",
        "preload_models": ["llama3.1:8b"],
        "max_loaded_models": 1,
        "num_parallel_requests": 1
    }
}

if PROFILE not in PROFILE_PRESETS:
    raise ValueError(f"Invalid PROFILE: {PROFILE}. Use one of: {list(PROFILE_PRESETS)}")

preset = PROFILE_PRESETS[PROFILE]
DEFAULT_MODEL = preset["default_model"]
PRELOAD_MODELS = preset["preload_models"]
MAX_LOADED_MODELS = preset["max_loaded_models"]
NUM_PARALLEL_REQUESTS = preset["num_parallel_requests"]

TERMINAL_PORT = 7681
OLLAMA_PORT = 11434

os.environ["OLLAMA_CONTEXT_LENGTH"] = "16384"
os.environ["OLLAMA_HOST"] = "0.0.0.0"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = str(MAX_LOADED_MODELS)
os.environ["OLLAMA_NUM_PARALLEL"] = str(NUM_PARALLEL_REQUESTS)

print("Ollama env configured")
print(f"- PROFILE={PROFILE}")
print(f"- DEFAULT_MODEL={DEFAULT_MODEL}")
print(f"- PRELOAD_MODELS={PRELOAD_MODELS}")
print(f"- OLLAMA_MAX_LOADED_MODELS={MAX_LOADED_MODELS}")
print(f"- OLLAMA_NUM_PARALLEL={NUM_PARALLEL_REQUESTS}")

In [ ]:
# Optional UI: select profile without editing code (run this before startup cells)
try:
    import ipywidgets as widgets
    from IPython.display import display
except Exception:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])
    import ipywidgets as widgets
    from IPython.display import display

profile_dropdown = widgets.Dropdown(
    options=["agente", "codigo", "geral", "imagem"],
    value=PROFILE,
    description="Profile:",
    layout=widgets.Layout(width="280px")
)
apply_button = widgets.Button(
    description="Apply profile",
    button_style="success",
    icon="check"
    )
output = widgets.Output()

def apply_profile(_):
    global PROFILE, DEFAULT_MODEL, PRELOAD_MODELS, MAX_LOADED_MODELS, NUM_PARALLEL_REQUESTS
    with output:
        output.clear_output()
        PROFILE = profile_dropdown.value
        preset = PROFILE_PRESETS[PROFILE]
        DEFAULT_MODEL = preset["default_model"]
        PRELOAD_MODELS = preset["preload_models"]
        MAX_LOADED_MODELS = preset["max_loaded_models"]
        NUM_PARALLEL_REQUESTS = preset["num_parallel_requests"]

        os.environ["OLLAMA_MAX_LOADED_MODELS"] = str(MAX_LOADED_MODELS)
        os.environ["OLLAMA_NUM_PARALLEL"] = str(NUM_PARALLEL_REQUESTS)

        print("Profile applied")
        print(f"- PROFILE={PROFILE}")
        print(f"- DEFAULT_MODEL={DEFAULT_MODEL}")
        print(f"- PRELOAD_MODELS={PRELOAD_MODELS}")
        print(f"- OLLAMA_MAX_LOADED_MODELS={MAX_LOADED_MODELS}")
        print(f"- OLLAMA_NUM_PARALLEL={NUM_PARALLEL_REQUESTS}")

apply_button.on_click(apply_profile)
display(widgets.HBox([profile_dropdown, apply_button]))
display(output)
print("Tip: choose a profile and click 'Apply profile' before running startup cells.")

In [ ]:
!apt-get update -y
!apt-get install -y curl wget lshw pciutils
!nvidia-smi
!nvcc --version

from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print(f"\nAvailable RAM: {ram_gb:.1f} GB")
print("High-RAM runtime" if ram_gb >= 20 else "Not a high-RAM runtime")

In [ ]:
# Install Ollama
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Install ttyd (web terminal)
!apt-get install -y ttyd || true
!if [ ! -x /usr/bin/ttyd ]; then wget -q https://github.com/tsl0922/ttyd/releases/latest/download/ttyd.x86_64 -O /usr/local/bin/ttyd && chmod +x /usr/local/bin/ttyd; fi
!ttyd --version

In [ ]:
import subprocess
import threading
import time
import requests

def start_ollama():
    subprocess.call(["ollama", "serve"])

ollama_thread = threading.Thread(target=start_ollama, daemon=True)
ollama_thread.start()

def wait_for_ollama(timeout=120):
    for i in range(timeout):
        try:
            r = requests.get(f"http://localhost:{OLLAMA_PORT}", timeout=2)
            if r.status_code in [200, 404]:
                print(f"Ollama is up after {i+1}s")
                return
        except requests.exceptions.RequestException:
            pass
        time.sleep(1)
    raise RuntimeError("Ollama did not start in time")

def pull_models(models):
    for model in models:
        print(f"Pulling {model}...")
        subprocess.check_call(["ollama", "pull", model])

wait_for_ollama()
pull_models(PRELOAD_MODELS or [DEFAULT_MODEL])
print("Initial pull done")

In [ ]:
# Optional: start one or more models and keep them warm in memory
from concurrent.futures import ThreadPoolExecutor, as_completed

# If empty, profile defaults will be used automatically.
OPTIONAL_START_MODEL = None
OPTIONAL_START_MODELS = []

def warm_model(model_name, prompt="reply with: ready", keep_alive="2h"):
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "options": {"num_predict": 8},
        "keep_alive": keep_alive
    }
    r = requests.post(f"http://localhost:{OLLAMA_PORT}/api/generate", json=payload, timeout=300)
    r.raise_for_status()
    return model_name

def start_models_parallel(models):
    if not models:
        print("No optional models selected. Skipping warm start.")
        return []

    started = []
    with ThreadPoolExecutor(max_workers=min(len(models), NUM_PARALLEL_REQUESTS)) as ex:
        futures = [ex.submit(warm_model, m) for m in models]
        for fut in as_completed(futures):
            model = fut.result()
            started.append(model)
            print(f"Warm and running: {model}")

    print("\nModels currently warmed:")
    for m in started:
        print(f"- {m}")
    return started

selected_models = []
if OPTIONAL_START_MODEL:
    selected_models.append(OPTIONAL_START_MODEL)
selected_models.extend(OPTIONAL_START_MODELS)

if not selected_models:
    selected_models = list(PRELOAD_MODELS)

# Remove duplicates while preserving order
selected_models = list(dict.fromkeys(selected_models))

RUNNING_MODELS = start_models_parallel(selected_models)

In [ ]:
import re

terminal_user = None
terminal_password = None

ttyd_proc = subprocess.Popen(
    [
        "ttyd",
        "-p", str(TERMINAL_PORT),
        "bash", "-lc", "cd /content && exec bash"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

terminal_tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{TERMINAL_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

terminal_url = None
for line in terminal_tunnel.stdout:
    print(line.strip())
    m = re.search(r"(https://.*\.trycloudflare\.com)", line)
    if m:
        terminal_url = m.group(1)
        break

if not terminal_url:
    raise RuntimeError("Could not create public terminal URL")

print("\nWeb terminal ready (NO password)")
print(f"URL: {terminal_url}")

In [ ]:
# Optional: also expose Ollama API publicly
ollama_tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{OLLAMA_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

ollama_url = None
for line in ollama_tunnel.stdout:
    print(line.strip())
    m = re.search(r"(https://.*\.trycloudflare\.com)", line)
    if m:
        ollama_url = m.group(1)
        break

if ollama_url:
    print(f"\nPublic URL for Ollama API: {ollama_url}")
else:
    raise RuntimeError("Could not create public URL for Ollama")

In [ ]:
# Model options and useful terminal commands for Google Colab T4
MODEL_OPTIONS = {
    "agente_geral_hermes": [
        "ollama pull nous-hermes2:10.7b",
        "ollama run nous-hermes2:10.7b"
    ],
    "codificacao": [
        "ollama pull qwen2.5-coder:7b",
        "ollama pull deepseek-coder:6.7b",
        "ollama run qwen2.5-coder:7b"
    ],
    "uso_geral": [
        "ollama pull llama3.1:8b",
        "ollama pull mistral:7b",
        "ollama run llama3.1:8b"
    ],
    "geracao_de_imagem": [
        "python3 -m pip install -q diffusers transformers accelerate safetensors",
        "python3 /content/generate_image_t4.py"
    ]
}

print(f"Active PROFILE: {PROFILE}")
print("\nChange PROFILE in the first cell to switch presets quickly.")
print("Available PROFILE values: agente, codigo, geral, imagem")

print("\nRecommended command options by use case:")
for category, commands in MODEL_OPTIONS.items():
    print(f"\n[{category}]")
    for cmd in commands:
        print(cmd)

print("\nRun multiple models in parallel on T4 (watch VRAM):")
print("ollama pull llama3.1:8b && ollama pull qwen2.5-coder:7b")
print("curl -s http://localhost:11434/api/generate -d '{\"model\":\"llama3.1:8b\",\"prompt\":\"ready\",\"stream\":false,\"keep_alive\":\"2h\"}'")
print("curl -s http://localhost:11434/api/generate -d '{\"model\":\"qwen2.5-coder:7b\",\"prompt\":\"ready\",\"stream\":false,\"keep_alive\":\"2h\"}'")
print("ollama ps")

In [ ]:
# Optional image model starter for T4 (save script and run from web terminal or notebook)
image_script = r'''
import torch
from diffusers import AutoPipelineForText2Image

model_id = "stabilityai/sdxl-turbo"
pipe = AutoPipelineForText2Image.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

prompt = "a cinematic cyberpunk city at sunset, ultra detailed, 4k"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
output_path = "/content/sdxl_t4_output.png"
image.save(output_path)
print(f"Image saved to {output_path}")
'''

with open("/content/generate_image_t4.py", "w", encoding="utf-8") as f:
    f.write(image_script)

print("Created: /content/generate_image_t4.py")
print("To run:")
print("python3 -m pip install -q diffusers transformers accelerate safetensors")
print("python3 /content/generate_image_t4.py")